这一章对应后端面试里非常典型的“中间件 + 服务通信 + 网络框架”部分。

如果前面分布式、高并发、微服务讲的是系统设计思路，那么这一章更偏工程落地：系统拆开以后，服务之间怎么通信，异步任务怎么处理，搜索能力怎么建设，高并发网络服务怎么写，Go 项目里怎么选框架。

这一章分成五条主线：

第一，消息队列。
面试官常问为什么用 MQ，MQ 解决什么问题，Kafka、RocketMQ、RabbitMQ 怎么选，消息如何保证不丢，为什么会重复，如何保证幂等，顺序消息怎么做，消息积压怎么排查，延迟消息和死信队列怎么用。Go 里要补充的是：consumer 一定要控制并发，不要无限起 goroutine；所有处理逻辑要带 context；消费成功后再 ack；失败要进入重试、死信或补偿。

第二，Elasticsearch。
ES 常问为什么不用 MySQL 做搜索，倒排索引是什么，分词器是什么，mapping 怎么设计，为什么是近实时，分片和副本有什么作用，深分页为什么慢，MySQL 和 ES 数据一致性怎么保证。Go 里要补充的是：ES 通常不是主存储，而是查询侧索引；写入链路要通过 binlog、MQ、Outbox 或 CDC 同步；业务要能接受搜索侧的短暂延迟。

第三，RPC。
RPC 的核心不是“远程调用看起来像本地调用”这么简单，而是协议、序列化、连接管理、服务发现、负载均衡、超时、重试、熔断、限流、链路追踪这一整套能力。Java 里常讲 Dubbo，Go 里常讲 gRPC、Kitex、Kratos、go-zero，或者公司内部 RPC 框架。

第四，gRPC。
gRPC 是 Go 微服务里非常高频的服务通信方案。它基于 HTTP/2，默认使用 Protobuf，支持一元调用和流式调用。面试重点包括 proto 定义、生成代码、metadata、deadline、status code、interceptor、stream、负载均衡、服务发现、与 REST 的区别。

第五，Go 网络框架。
Java 里问 Netty，核心是 NIO、Reactor、EventLoop、Pipeline、编解码、粘包拆包、心跳。Go 里不一定需要一个“Go 版 Netty”，因为 Go runtime 已经用 netpoller + goroutine 封装了高并发网络 IO。Go 业务开发常用 net/http、Gin、Echo、Fiber、Hertz；微服务框架常见 Kratos、go-zero、Kitex、gRPC；极端长连接或事件驱动场景才可能考虑 gnet 这类框架。

本章最终要形成一句话：

> MQ 解决异步、解耦和削峰；ES 解决搜索和复杂检索；RPC/gRPC 解决服务间高效通信；Go 网络框架解决请求接入、路由、中间件和治理。真正面试时不能只背概念，而要讲出可靠性、幂等、超时、重试、限流、监控、链路追踪这些工程闭环。

---

# 一、消息队列基础

## 【文本块 2】Q：项目里为什么要引入消息队列？

消息队列最核心的价值有三个：

第一，解耦。
上游系统只负责把事件发布出去，下游系统独立消费。比如订单创建成功后，通知服务、积分服务、优惠券服务、风控服务都可能需要感知这个事件。如果不用 MQ，订单服务就要同步调用多个下游，耦合很重。用了 MQ 之后，订单服务只发一条 `order.created` 消息，下游谁关心谁订阅。

第二，削峰填谷。
高峰流量先进入 MQ，下游系统按照自己的消费能力慢慢处理，避免瞬时流量直接打爆数据库或核心服务。比如秒杀下单，入口层 Redis 预扣库存后，把成功请求写 MQ，订单服务异步消费落库。

第三，异步化。
一些不需要主链路同步完成的事情，可以异步执行，从而降低接口 RT。比如发短信、发邮件、发积分、写埋点、同步搜索索引、生成报表。

但 MQ 不是免费午餐。引入 MQ 之后，系统会从“同步成功/失败”变成“最终一致”，同时引入消息丢失、重复消费、乱序、积压、死信、补偿、可观测性等问题。

面试里可以这样回答：

> MQ 主要解决解耦、削峰和异步化。比如下单成功后，订单服务不需要同步调用通知、积分、埋点等多个下游，而是发布订单事件，下游异步消费。但用 MQ 之后必须处理可靠投递、消费幂等、失败重试、死信补偿和消息积压监控，否则只是把复杂度从同步链路转移到了异步链路。

---

## 【文本块 3】Q：消息队列的基本模型是什么？

一个最基础的 MQ 模型一般包括：

Producer：生产者，负责发送消息。
Broker：消息服务器，负责接收、存储、分发消息。
Topic 或 Queue：消息的逻辑分类。
Consumer：消费者，负责处理消息。
Consumer Group：消费组，用来实现负载均衡和广播语义。
Offset 或 Ack：消费进度或消费确认机制。

不同 MQ 的术语略有差异。

Kafka 里核心是 Topic、Partition、Consumer Group、Offset。
RocketMQ 里核心是 Topic、Message Queue、Producer、Consumer、NameServer、Broker。
RabbitMQ 里核心是 Exchange、Queue、Binding、Routing Key、Consumer Ack。

面试里不要只背名词，要讲清楚：

> MQ 的本质是把生产者和消费者通过一个可靠的中间层隔开。生产者把消息交给 Broker，Broker 按主题、队列或路由规则存储和投递，消费者处理成功后确认消费进度。

---

## 【代码块 1】Go 中用接口抽象 MQ Producer

```go
package main

import (
	"context"
	"encoding/json"
	"fmt"
)

type Message struct {
	Topic string
	Key   string
	Body  []byte
}

type Producer interface {
	Send(ctx context.Context, msg Message) error
}

type OrderCreatedEvent struct {
	OrderID int64  `json:"order_id"`
	UserID  int64  `json:"user_id"`
	Status  string `json:"status"`
}

type OrderService struct {
	producer Producer
}

func (s *OrderService) PublishOrderCreated(ctx context.Context, event OrderCreatedEvent) error {
	body, err := json.Marshal(event)
	if err != nil {
		return fmt.Errorf("marshal order created event: %w", err)
	}

	msg := Message{
		Topic: "order.created",
		Key:   fmt.Sprintf("%d", event.OrderID),
		Body:  body,
	}

	if err := s.producer.Send(ctx, msg); err != nil {
		return fmt.Errorf("send order created event: %w", err)
	}

	return nil
}
```

---

## 【文本块 4】代码解释

这里没有把业务代码直接绑定到 Kafka、RocketMQ 或 RabbitMQ 的具体 client。

业务层只依赖一个 Producer 接口。
具体是 KafkaProducer、RocketMQProducer 还是 RabbitMQProducer，由基础设施层实现。

这样做的好处是：

第一，业务逻辑更干净。
第二，方便单元测试，可以 mock Producer。
第三，后续替换 MQ 或调整发送逻辑时，影响范围更小。
第四，可以在统一 Producer 实现里加入 trace、metrics、retry、timeout、日志等横切能力。

这就是 Go 项目里比较推荐的写法：面向接口组织依赖，而不是让业务层到处散落具体中间件调用。

---

# 二、消息可靠性

## 【文本块 5】Q：MQ 如何保证消息不丢失？

消息可靠性要分三段看。

第一段，生产者到 Broker。
生产者发送消息后，不能只管 fire-and-forget。要确认 Broker 已经收到。Kafka 有 ack 机制，RabbitMQ 有 Publisher Confirm，RocketMQ 也有发送结果。生产端还要处理发送失败、超时、重试和本地消息表。

第二段，Broker 存储。
Broker 收到消息后，要考虑持久化、副本、刷盘策略、主从复制。如果 Broker 还没持久化就宕机，消息可能丢。不同 MQ 在可靠性和吞吐之间都有配置取舍。

第三段，Broker 到 Consumer。
消费者处理成功后才能 ack 或提交 offset。如果业务还没处理完就提前确认，一旦消费者崩溃，消息就丢了。消费者失败时要重试，重试多次仍失败要进死信队列或补偿流程。

标准回答可以这样组织：

> MQ 可靠性不能只看 Broker，要分生产端、Broker、消费端三段。生产端要确认发送成功，必要时用本地消息表；Broker 端要持久化和副本；消费端要业务成功后再 ack，并且配合重试、死信和幂等。

---

## 【文本块 6】追问：只要 MQ 开了持久化，消息就一定不丢吗？

不一定。

持久化只是解决 Broker 内部存储的一部分问题。

还有很多可能丢消息的地方：

生产者发送失败但没有重试。
生产者以为发送成功，其实网络超时结果未知。
Broker 还没刷盘就宕机。
主从复制还没完成，主节点挂了。
消费者提前 ack，业务处理失败。
消费者处理成功但写数据库失败。
业务代码吞掉异常，仍然提交 offset。

所以消息可靠性一定是端到端设计，不是打开某个配置就万事大吉。

---

## 【文本块 7】Q：什么是本地消息表 / Outbox Pattern？

本地消息表的核心是：业务数据和待发送消息在同一个数据库事务里提交。

比如订单创建：

1. 开启本地事务
2. 写订单表
3. 写 outbox 消息表
4. 提交事务
5. 后台 relay 扫描 outbox 表并发送 MQ
6. 发送成功后更新 outbox 状态

这样可以保证：

> 只要订单写库成功，就一定有一条消息等待发送。

它解决的是“业务写库成功，但 MQ 发送失败”这个经典问题。

---

## 【代码块 2】Outbox relay 简化示例

```go
package main

import (
	"context"
	"database/sql"
	"fmt"
	"time"
)

type OutboxRecord struct {
	ID    int64
	Topic string
	Key   string
	Body  []byte
}

type OutboxProducer interface {
	Send(ctx context.Context, msg Message) error
}

func RelayOutboxOnce(ctx context.Context, db *sql.DB, producer OutboxProducer) error {
	rows, err := db.QueryContext(ctx, `
		SELECT id, topic, msg_key, body
		FROM outbox
		WHERE status = 'NEW'
		ORDER BY id
		LIMIT 100
	`)
	if err != nil {
		return fmt.Errorf("query outbox: %w", err)
	}
	defer rows.Close()

	var records []OutboxRecord
	for rows.Next() {
		var r OutboxRecord
		if err := rows.Scan(&r.ID, &r.Topic, &r.Key, &r.Body); err != nil {
			return fmt.Errorf("scan outbox: %w", err)
		}
		records = append(records, r)
	}

	for _, r := range records {
		sendCtx, cancel := context.WithTimeout(ctx, 3*time.Second)
		err := producer.Send(sendCtx, Message{
			Topic: r.Topic,
			Key:   r.Key,
			Body:  r.Body,
		})
		cancel()

		if err != nil {
			// 真实项目里这里要记录失败次数、下次重试时间、错误原因。
			continue
		}

		_, err = db.ExecContext(ctx, `
			UPDATE outbox
			SET status = 'SENT', sent_at = NOW()
			WHERE id = ?
		`, r.ID)
		if err != nil {
			return fmt.Errorf("mark outbox sent: %w", err)
		}
	}

	return nil
}
```

---

## 【文本块 8】代码解释

这段代码体现的是 Outbox relay 的核心流程：

先扫描状态为 NEW 的消息。
逐条发送到 MQ。
发送成功后把状态更新为 SENT。

真实项目里还要补充：

* 多实例 relay 抢占，避免重复发送
* 失败次数和下次重试时间
* 发送超时控制
* trace_id 透传
* metrics 监控
* 死信或人工处理
* 定期对账
* 消费端幂等

注意：Outbox 不能保证消息只发送一次。
比如消息已经发送成功，但更新 outbox 状态失败，relay 下次可能再次发送。
所以消费端仍然必须做幂等。

---

# 三、重复消费与幂等

## 【文本块 9】Q：MQ 为什么会重复消费？

重复消费是 MQ 系统里的正常现象。

常见原因：

消费者处理成功了，但 ack 或 offset 提交失败。
消费者处理过程中重启了。
Broker 没收到确认，以为消费失败，重新投递。
生产者发送超时后重试，导致同一业务消息发了多次。
消费者组 rebalance 时，部分消息被重新分配。
Outbox relay 发送成功但更新状态失败，后续再次发送。

所以工程里不要追求“绝对不重复”，而要设计成：

> MQ 至少一次投递，业务消费必须幂等。

---

## 【文本块 10】Q：消费端如何保证幂等？

常见做法有几种。

第一，唯一索引。
比如消息里有 event_id 或 order_id，对消费结果表建唯一索引。重复插入直接失败，然后返回成功。

第二，消费记录表。
处理前先插入消费记录，状态为 PROCESSING；处理成功后改成 SUCCESS。重复消息来了，发现已经 SUCCESS 就跳过。

第三，业务状态机。
比如订单只能从 CREATED 变成 PAID，不能重复变成 PAID 并重复发货。

第四，Redis SETNX。
用 message_id 作为 key，抢到执行权才处理。但 Redis 幂等要注意 TTL、失败恢复和最终一致。

第五，下游接口本身幂等。
比如调用支付、库存、发券接口时，带幂等号。

面试里可以这样回答：

> MQ 重复投递是常态，所以消费端必须幂等。核心写操作我一般会用业务唯一键或消费记录表兜底；状态流转用状态机限制；调用下游时继续传幂等号。不要依赖“MQ 不重复”来保证业务正确性。

---

## 【代码块 3】消费记录表实现幂等的简化示例

```go
package main

import (
	"context"
	"database/sql"
	"fmt"
)

func ConsumeOrderEvent(ctx context.Context, db *sql.DB, eventID string, orderID int64) error {
	tx, err := db.BeginTx(ctx, nil)
	if err != nil {
		return fmt.Errorf("begin tx: %w", err)
	}
	defer tx.Rollback()

	_, err = tx.ExecContext(ctx, `
		INSERT INTO consumed_messages(event_id, status)
		VALUES (?, 'PROCESSING')
	`, eventID)
	if err != nil {
		// 实际项目里要判断是否 duplicate key。
		// 如果已经消费成功，直接返回 nil。
		return fmt.Errorf("insert consumed message: %w", err)
	}

	_, err = tx.ExecContext(ctx, `
		UPDATE orders
		SET status = 'PAID'
		WHERE id = ? AND status = 'CREATED'
	`, orderID)
	if err != nil {
		return fmt.Errorf("update order status: %w", err)
	}

	_, err = tx.ExecContext(ctx, `
		UPDATE consumed_messages
		SET status = 'SUCCESS'
		WHERE event_id = ?
	`, eventID)
	if err != nil {
		return fmt.Errorf("mark message success: %w", err)
	}

	if err := tx.Commit(); err != nil {
		return fmt.Errorf("commit tx: %w", err)
	}

	return nil
}
```

---

## 【文本块 11】代码解释

这段代码有两个幂等点：

第一，`consumed_messages.event_id` 应该有唯一索引。
同一条消息重复来时，插入会失败，业务可以判断是否已经处理过。

第二，订单状态更新带条件：

```sql
WHERE id = ? AND status = 'CREATED'
```

这能避免已经 PAID 的订单被重复处理。

真实项目里还要区分：

* PROCESSING 超时怎么办
* SUCCESS 直接跳过
* FAILED 是否允许重试
* 消费记录是否归档
* 消息 ID 是全局唯一还是业务唯一
* 消费失败是否进入死信

---

# 四、顺序消息

## 【文本块 12】Q：MQ 如何保证消息顺序？

先区分两种顺序。

第一，全局顺序。
所有消息都严格按发送顺序消费。这种吞吐很低，因为基本只能单队列、单消费者。

第二，局部顺序。
同一个业务 key 下有序，比如同一个订单的状态变更有序，同一个用户的操作有序，不同订单之间可以并行。

真实业务里大多数需要的是局部顺序，而不是全局顺序。

常见做法：

* 按业务 key 选择同一个 partition / queue
* 同一个 partition / queue 由同一个消费者线程顺序处理
* 消费失败时要暂停或重试，避免后续消息越过失败消息
* 消费逻辑不能在内部乱起 goroutine 打乱顺序

例如订单状态流转：

```text
ORDER_CREATED -> ORDER_PAID -> ORDER_SHIPPED -> ORDER_FINISHED
```

同一个 order_id 的消息必须发到同一个分区，才能保证顺序。

面试里可以说：

> 顺序消息通常不是全局顺序，而是按业务 key 的局部顺序。核心是让同一个 key 的消息进入同一个分区或队列，并由同一个消费单元顺序处理。代价是并行度受 key 分布影响，热点 key 会成为瓶颈。

---

## 【文本块 13】追问：顺序消息和吞吐有什么矛盾？

顺序要求越强，并行度越低。

如果要求全局顺序，所有消息只能进一个队列，一个消费者顺序处理，吞吐很差。

如果要求局部顺序，可以按 order_id、user_id、device_id 分区，不同 key 并行处理，同一个 key 内部顺序处理，吞吐会好很多。

所以工程里要尽量把全局顺序降级成局部顺序。

---

# 五、消息积压排查

## 【文本块 14】Q：消息积压一般怎么排查？

消息积压通常从三端看。

第一，上游生产太快。
比如大促流量、批量任务、重试风暴，导致生产速度远大于消费速度。

第二，消费者消费太慢。
可能是业务逻辑太重、DB 慢、RPC 慢、锁竞争、单条消息处理耗时太长。

第三，消费并行度不足。
Kafka partition 数太少，消费者实例太少；RabbitMQ 单队列成为瓶颈；RocketMQ queue 数不够。

第四，配置不合理。
比如 prefetch 太大或太小，batch size 不合理，ack 太慢，消费者线程池太小。

第五，下游依赖故障。
消费者本身没问题，但它依赖的 MySQL、Redis、RPC 慢了，导致整体消费能力下降。

面试里可以这样回答：

> 排查消息积压，我会先看生产速率和消费速率，再看消费者单条耗时、错误率、重试次数、下游 DB/RPC RT，最后看 partition/queue 数、消费者实例数和并发配置。解决方式包括扩消费者、提高并行度、优化消费逻辑、批量处理、限流上游、降级非核心逻辑，必要时做死信和补偿。

---

## 【代码块 4】Go 消费者 worker pool 示例

```go
package main

import (
	"context"
	"fmt"
	"sync"
	"time"
)

type ConsumerMessage struct {
	ID    string
	Topic string
	Key   string
	Body  []byte
}

type MessageHandler interface {
	Handle(ctx context.Context, msg ConsumerMessage) error
}

func StartWorkerPool(
	ctx context.Context,
	workerNum int,
	msgCh <-chan ConsumerMessage,
	handler MessageHandler,
) {
	var wg sync.WaitGroup

	for i := 0; i < workerNum; i++ {
		workerID := i

		wg.Add(1)
		go func() {
			defer wg.Done()

			for {
				select {
				case <-ctx.Done():
					fmt.Println("worker stopped:", workerID)
					return

				case msg, ok := <-msgCh:
					if !ok {
						return
					}

					handleCtx, cancel := context.WithTimeout(ctx, 5*time.Second)
					err := handler.Handle(handleCtx, msg)
					cancel()

					if err != nil {
						// 实际项目里：不要简单打印。
						// 应该区分可重试错误、不可重试错误、死信、告警。
						fmt.Println("handle message failed:", err)
						continue
					}

					// 对 Kafka：这里可以提交 offset。
					// 对 RabbitMQ：这里可以 ack。
					// 对 RocketMQ：这里可以返回消费成功。
				}
			}
		}()
	}

	go func() {
		<-ctx.Done()
		wg.Wait()
	}()
}
```

---

## 【文本块 15】代码解释

这段代码体现 Go 消费者的一个重要原则：

> 不要 MQ 来一条消息就无限开一个 goroutine。

无限 goroutine 会导致：

* 内存暴涨
* DB 被打爆
* RPC 下游被打爆
* 连接池耗尽
* goroutine 泄漏
* 消费失败更难控制

更好的方式是 worker pool 控制并发，让消费者吞吐和下游承载能力匹配。

真实项目里还要加：

* 消费失败重试策略
* panic recover
* metrics
* trace
* 死信队列
* 幂等
* 批量消费
* 优雅停机
* backpressure

---

# 六、Kafka、RocketMQ、RabbitMQ 选型

## 【文本块 16】Q：Kafka、RocketMQ、RabbitMQ 怎么选？

可以用一句话先概括：

Kafka 更偏高吞吐日志流和数据流。
RocketMQ 更偏核心业务消息，顺序、延迟、事务消息能力比较完整。
RabbitMQ 更偏灵活路由和传统业务队列，适合中后台任务、通知分发、审批流等场景。

更细一点：

Kafka 的优势：

* 高吞吐
* 分区并行
* 顺序写磁盘
* Page Cache
* 零拷贝
* 批量发送
* 压缩
* 消费者生态和大数据链路成熟

适合：

* 日志采集
* 埋点流
* 数据管道
* 大数据分析
* 流式处理
* 高吞吐事件流

RocketMQ 的优势：

* 业务消息能力较完整
* 顺序消息
* 延迟消息
* 事务消息
* 重试和死信机制
* 适合订单、支付、库存这类业务链路

适合：

* 电商交易
* 订单状态流转
* 支付回调
* 库存扣减
* 业务事件通知

RabbitMQ 的优势：

* Exchange 路由模型灵活
* AMQP 协议成熟
* Direct/Fanout/Topic/Headers 路由
* 管理和使用门槛相对低
* 适合规则路由复杂但吞吐不是极致的业务

适合：

* 通知分发
* 审批流
* 后台任务
* 工作队列
* 传统业务系统异步解耦

面试里可以这样回答：

> MQ 选型不是看谁功能最多，而是看业务诉求。日志流和大吞吐优先 Kafka；订单、支付、库存这类核心业务消息可以考虑 RocketMQ；需要灵活路由、中后台异步任务、团队使用成本低的场景可以考虑 RabbitMQ。无论选哪个，都要重点设计可靠投递、消费幂等、积压监控和死信补偿。

---

# 七、RabbitMQ 高频点

## 【文本块 17】Q：RabbitMQ 的 Exchange 有哪些类型？

常见四种：

Direct Exchange：
按精确 routing key 路由。适合定向投递。

Fanout Exchange：
广播到所有绑定队列，忽略 routing key。适合广播通知。

Topic Exchange：
支持通配符匹配，适合按主题和规则灵活路由。

Headers Exchange：
按消息头匹配，实际业务里没有前三种常见。

RabbitMQ 的强项就是 Exchange 模型。Producer 不是直接把消息发到 Queue，而是先发到 Exchange，再由 Exchange 根据 Binding 和 Routing Key 路由到 Queue。

---

## 【文本块 18】Q：RabbitMQ 如何保证可靠性？

RabbitMQ 可靠性也可以分三段：

生产端：

* 开启 Publisher Confirm
* 必要时配合 Return 机制处理无法路由消息
* 发送失败要重试或落本地表

Broker 端：

* Exchange 持久化
* Queue 持久化
* Message 持久化
* 高可用队列，比如 Quorum Queue

消费端：

* 使用手动 ack
* 业务处理成功后再 ack
* 失败时 nack/reject
* 多次失败进入死信队列
* 消费端幂等

一句话：

> RabbitMQ 可靠性标准回答是：生产端 Confirm，Broker 端持久化，消费端手动 ack，再配合死信和幂等。

---

## 【文本块 19】Q：RabbitMQ 的 prefetch 是什么？

prefetch 表示消费者一次最多可以预取多少条尚未确认的消息。

它的作用是消费端流控。

如果 prefetch 太大，某个消费者可能一次拿太多消息，但处理不过来，导致消息压在消费者本地，其他消费者拿不到，形成不公平。

如果 prefetch 太小，消费者每次拿太少，吞吐可能起不来。

所以 prefetch 要结合单条消息处理耗时、消费者数量、业务并发能力来调整。

---

# 八、Kafka 高频点

## 【文本块 20】Q：Kafka 为什么吞吐高？

Kafka 高吞吐主要靠几个设计：

第一，顺序写磁盘。
Kafka 把消息追加写入日志文件，顺序写比随机写快很多。

第二，Page Cache。
Kafka 大量利用操作系统页缓存，写入先进入内存，由 OS 异步刷盘。

第三，零拷贝。
读取数据发送给消费者时，可以减少用户态和内核态之间的数据拷贝。

第四，批量发送。
Producer 可以攒一批消息再发，减少网络请求次数。

第五，压缩。
消息压缩可以减少网络传输和磁盘占用。

第六，分区并行。
Topic 可以拆成多个 Partition，生产和消费都能并行。

第七，顺序读写 + 稀疏索引。
通过 offset 和索引快速定位消息。

面试里可以这样回答：

> Kafka 高吞吐不是单点优化，而是顺序写、Page Cache、零拷贝、批量、压缩和分区并行共同作用的结果。

---

## 【文本块 21】Q：Kafka 如何保证顺序？

Kafka 只能保证单个 Partition 内的消息顺序。

如果同一个业务 key 的消息要有序，就必须让它们进入同一个 Partition。

比如同一个 order_id 的订单状态消息，用 order_id 作为 key。Producer 根据 key 选择 Partition，这样同一个订单的消息会进入同一个分区，消费者按 offset 顺序消费。

但跨 Partition 不保证全局顺序。

所以面试里说：

> Kafka 保证的是分区内有序，不保证全局有序。业务要顺序，就要按业务 key 分区，把同一个 key 的消息发到同一个 Partition。

---

## 【文本块 22】Q：Kafka 的 offset 是什么？

offset 是消息在 Partition 内的位置。

Consumer 消费消息后，需要提交 offset，表示“我已经消费到哪里了”。

提交 offset 有两种思路：

自动提交：简单，但可能业务还没处理完 offset 就提交了，导致消息丢。
手动提交：业务处理成功后再提交，更可靠，但代码复杂些。

工程里核心消息通常更推荐手动提交 offset。

不过手动提交也不能保证不重复。
如果业务处理成功，但提交 offset 失败，消息会被重新消费，所以仍然要幂等。

---

# 九、RocketMQ 高频点

## 【文本块 23】Q：RocketMQ 的核心概念有哪些？

RocketMQ 常见核心概念：

Producer：生产者。
Consumer：消费者。
NameServer：路由中心，记录 Topic 和 Broker 的路由信息。
Broker：真正存储消息、提供读写服务的节点。
Topic：消息主题。
Message Queue：Topic 下的逻辑队列。
Consumer Group：消费组。
Tag：更细粒度的消息分类。

可以这样理解：

NameServer 解决“消息发到哪里”。
Broker 解决“消息存在哪里、怎么投递”。
Queue 解决“并行度和局部有序”。
Consumer Group 解决“一组消费者如何分摊消费”。

---

## 【文本块 24】Q：RocketMQ 事务消息解决什么问题？

RocketMQ 事务消息主要解决：

> 本地事务和发送消息之间的一致性问题。

典型流程：

1. 发送半消息
2. Broker 暂时不让消费者看到
3. 执行业务本地事务
4. 本地事务成功后提交消息
5. 本地事务失败后回滚消息
6. 如果 Producer 没返回结果，Broker 会回查本地事务状态

但要注意：

事务消息不是分布式事务万能解法。
它解决的是“本地事务和消息可见性”的一致性。
消费者仍然可能重复消费，所以消费端仍然要幂等。

面试里可以这样回答：

> RocketMQ 事务消息能保证本地事务成功后消息最终可见，本地事务失败则消息不可见。但它不能替代消费端幂等，因为消息投递和消费仍然可能重试。

---

# 十、Elasticsearch 基础

## 【文本块 25】Q：什么是 Elasticsearch？适合解决什么问题？

Elasticsearch 是一个分布式全文检索和分析引擎。

它最擅长解决：

* 全文搜索
* 模糊搜索
* 复杂条件过滤
* 海量文档检索
* 聚合分析
* 日志检索
* 商品搜索
* 文章搜索
* 运营报表查询

一句话：

> ES 的核心价值，是把“海量数据 + 模糊搜索 + 多条件检索 + 聚合分析”这类关系型数据库不擅长的查询变得高效。

它通常不是主数据库，而是查询侧索引。

比如商品系统中：

MySQL 存商品主数据。
ES 存商品搜索索引。
用户搜索关键词、筛选品牌、排序价格、聚合分类时走 ES。

---

## 【文本块 26】Q：为什么不用 MySQL 做搜索？

MySQL 可以做简单精确查询，但不适合复杂搜索。

比如：

```sql
WHERE title LIKE '%手机%'
```

这种前置模糊匹配很难有效利用普通 B+ 树索引，数据量大时性能会差。

而搜索场景通常还需要：

* 分词
* 相关性评分
* 多字段匹配
* 高亮
* 拼写纠错
* 同义词
* 复杂过滤
* 聚合统计
* 海量日志检索

这些不是 MySQL 的强项。

ES 的倒排索引更适合全文检索。
所以面试里可以说：

> MySQL 更适合事务型存储和精确查询，ES 更适合搜索和分析。二者不是替代关系，而是主存储和搜索索引的关系。

---

## 【文本块 27】Q：倒排索引是什么？

倒排索引可以理解成：

> 从“词”到“文档”的映射。

普通正向索引是：

```text
文档1 -> [苹果, 手机, 便宜]
文档2 -> [华为, 手机, 拍照]
文档3 -> [苹果, 电脑, 办公]
```

倒排索引是：

```text
苹果 -> [文档1, 文档3]
手机 -> [文档1, 文档2]
华为 -> [文档2]
电脑 -> [文档3]
```

当用户搜索“苹果手机”时，ES 会先分词，然后通过倒排索引快速找到包含这些词的文档，再根据相关性评分、过滤条件、排序规则返回结果。

面试里可以这样回答：

> 倒排索引就是从 term 到 document list 的映射。它让全文搜索不用逐行扫描所有文档，而是先根据关键词定位候选文档，再做相关性计算和排序。

---

## 【文本块 28】Q：分词器是什么？

分词器负责把文本切成一个个 term。

例如：

```text
我喜欢苹果手机
```

可能被分成：

```text
我 / 喜欢 / 苹果 / 手机
```

英文分词相对简单，按空格和标点就能处理很多场景。
中文没有天然空格，所以分词器很关键。

分词器通常包括：

* character filter
* tokenizer
* token filter

会做：

* 切词
* 转小写
* 去停用词
* 同义词扩展
* 词干提取
* 拼音转换

面试里要注意：

> ES 搜索结果不准，很多时候不是 ES 不行，而是 mapping、分词器、字段类型、查询 DSL 设计不合理。

---

## 【文本块 29】Q：mapping 怎么设计？

mapping 类似数据库表结构，定义字段类型和索引方式。

常见字段类型：

keyword：不分词，适合精确匹配、聚合、排序。
text：分词，适合全文搜索。
long/integer/double：数值。
date：时间。
boolean：布尔。
object/nested：对象结构。

举例：

商品标题适合 text，用于全文搜索。
商品 ID、品牌 ID、状态适合 keyword 或 long，用于过滤。
价格适合 long 或 double，用于排序和范围查询。
创建时间适合 date。
标签如果需要精确筛选，可以用 keyword。

面试里可以说：

> ES mapping 设计要根据查询场景来定。需要全文搜索的字段用 text，需要精确过滤、聚合、排序的字段用 keyword 或数值类型。不要所有字段都默认 text，也不要把会排序聚合的字段只建成 text。

---

## 【代码块 5】ES 商品索引 mapping 示例

```go
package main

import (
	"encoding/json"
	"fmt"
)

func main() {
	mapping := map[string]any{
		"mappings": map[string]any{
			"properties": map[string]any{
				"product_id": map[string]any{
					"type": "keyword",
				},
				"title": map[string]any{
					"type":     "text",
					"analyzer": "standard",
				},
				"brand_id": map[string]any{
					"type": "keyword",
				},
				"price": map[string]any{
					"type": "long",
				},
				"status": map[string]any{
					"type": "keyword",
				},
				"created_at": map[string]any{
					"type": "date",
				},
			},
		},
	}

	b, _ := json.MarshalIndent(mapping, "", "  ")
	fmt.Println(string(b))
}
```

---

## 【文本块 30】代码解释

这个 mapping 里：

product_id 用 keyword，因为它用于精确匹配。
title 用 text，因为它用于全文搜索。
brand_id 和 status 用 keyword，因为它们通常用于过滤和聚合。
price 用 long，因为它需要范围查询和排序。
created_at 用 date，因为它需要按时间过滤或排序。

实际项目里还要考虑：

* 中文分词器
* 多字段索引，比如 title 同时有 text 和 keyword 子字段
* 是否需要 nested
* 是否开启 doc_values
* 索引模板
* 分片数
* 生命周期管理
* 字段数量爆炸

---

# 十一、ES 写入、查询与近实时

## 【文本块 31】Q：为什么 ES 是近实时？

ES 写入文档后，不是每次写入都立刻对搜索可见。

ES 会把内存中的数据定期 refresh，生成新的 segment，之后搜索才能看到这些数据。

默认 refresh 间隔通常是秒级，所以 ES 被称为近实时搜索。

这意味着：

> 写入 ES 成功，不等于用户下一毫秒搜索一定能搜到。

对于普通搜索业务，秒级延迟通常可以接受。
但如果业务要求强一致查询，就不应该把 ES 当主库。

---

## 【文本块 32】Q：MySQL 和 ES 数据一致性怎么保证？

常见方案有三种。

第一，同步双写。
业务写 MySQL 后，同步写 ES。简单，但强耦合，ES 慢会拖慢主链路，失败处理复杂。

第二，MQ 异步同步。
业务写 MySQL 后发消息，消费者异步更新 ES。主链路快，但有最终一致延迟，必须处理消息可靠性和幂等。

第三，binlog / CDC 同步。
监听 MySQL binlog，把数据变更同步到 ES。侵入业务小，但链路复杂，需要处理乱序、延迟、回放、字段转换。

工程里更推荐把 MySQL 当主存储，ES 当查询索引。
主链路只保证 MySQL 正确，ES 通过 MQ、Outbox 或 CDC 最终同步。

面试里可以说：

> 我不会把 ES 当主库。MySQL 是事实源，ES 是搜索索引。数据同步一般通过 MQ、Outbox 或 binlog CDC 做最终一致。查询结果允许短暂延迟，异常时通过重试、补偿和重建索引修复。

---

## 【代码块 6】Go 中构造 ES 查询 DSL 示例

```go
package main

import (
	"encoding/json"
	"fmt"
)

func main() {
	query := map[string]any{
		"query": map[string]any{
			"bool": map[string]any{
				"must": []any{
					map[string]any{
						"match": map[string]any{
							"title": "苹果手机",
						},
					},
				},
				"filter": []any{
					map[string]any{
						"term": map[string]any{
							"status": "ON_SALE",
						},
					},
					map[string]any{
						"range": map[string]any{
							"price": map[string]any{
								"gte": 1000,
								"lte": 8000,
							},
						},
					},
				},
			},
		},
		"sort": []any{
			map[string]any{
				"price": "asc",
			},
		},
		"from": 0,
		"size": 20,
	}

	b, _ := json.MarshalIndent(query, "", "  ")
	fmt.Println(string(b))
}
```

---

## 【文本块 33】代码解释

这个查询表示：

标题匹配“苹果手机”。
状态必须是 ON_SALE。
价格在 1000 到 8000 之间。
按价格升序。
返回第一页 20 条。

这里要区分 must 和 filter：

must 会参与相关性评分。
filter 更适合精确过滤，通常可以缓存，不参与评分。

面试里可以说：

> 搜索条件里需要影响相关性的用 must/match；精确筛选条件用 filter/term/range。这样语义更清楚，性能也更好。

---

# 十二、ES 分片、副本、深分页

## 【文本块 34】Q：ES 的 shard 和 replica 是什么？

shard 是主分片，用来把一个索引的数据拆到多个分片上，提升容量和并行查询能力。

replica 是副本分片，用来提高可用性和读扩展能力。

比如一个 index 有 3 个 primary shard，每个 shard 有 1 个 replica，那么总共有 6 个 shard 副本实例。

主分片负责写入。
副本分片可以参与查询。
主分片挂了，副本可以提升为主分片。

面试里可以这样回答：

> shard 解决数据水平拆分和并行查询，replica 解决高可用和读扩展。分片不是越多越好，分片太多会增加集群管理开销、内存开销和查询协调成本。

---

## 【文本块 35】Q：ES 深分页为什么慢？

传统分页：

```json
{
  "from": 10000,
  "size": 20
}
```

意味着 ES 需要在各个分片上取出前 10020 条候选结果，再汇总排序后丢掉前 10000 条，只返回 20 条。

from 越大，浪费越严重。

解决方案：

第一，限制深分页。
业务上不允许翻到特别深。

第二，search_after。
基于上一页最后一条记录的排序值继续查下一页，适合游标分页。

第三，scroll。
适合大批量导出，不适合实时用户搜索。

第四，离线导出。
后台报表不要直接压在线 ES 查询。

面试里可以说：

> ES 深分页慢，是因为它要在多个分片上取大量候选结果再全局排序，from 越大浪费越严重。线上搜索一般限制页数，或者用 search_after 做游标分页。

---

# 十三、RPC 基础

## 【文本块 36】Q：什么是 RPC？

RPC 是 Remote Procedure Call，远程过程调用。

它的目标是让调用远程服务像调用本地函数一样简单。

比如本地调用：

```go
user, err := userService.GetUser(ctx, userID)
```

RPC 调用看起来也类似：

```go
user, err := userClient.GetUser(ctx, &GetUserRequest{UserID: userID})
```

但底层其实发生了很多事情：

1. 调用方把方法名、参数、metadata 编码
2. 序列化成字节流
3. 通过网络发送到服务端
4. 服务端解码请求
5. 找到对应方法执行
6. 把响应序列化
7. 通过网络返回
8. 客户端反序列化成对象

RPC 框架还会提供：

* 服务发现
* 负载均衡
* 连接池
* 超时控制
* 重试
* 熔断
* 限流
* 序列化
* 协议
* 拦截器
* 链路追踪
* 指标监控

面试里可以这样回答：

> RPC 不只是网络通信。它是服务间调用的一整套框架能力，包括协议、序列化、连接管理、服务发现、负载均衡、超时、重试、容错和治理。

---

## 【文本块 37】Q：RPC 和 HTTP REST 有什么区别？

HTTP REST 更偏资源模型，接口通常是 URL + Method + JSON。

例如：

```text
GET /users/1001
POST /orders
```

RPC 更偏方法调用模型。

例如：

```text
UserService.GetUser
OrderService.CreateOrder
```

常见区别：

REST：

* 通用性强
* 浏览器和网关友好
* 调试方便
* JSON 可读性好
* 适合对外开放 API

RPC：

* IDL 定义接口更严格
* 二进制协议性能更好
* 强类型生成代码
* 更适合内部服务调用
* 治理能力通常更细

面试里可以说：

> 对外 API 我更倾向 REST/HTTP，因为兼容性和可调试性好；内部服务高频调用可以用 RPC/gRPC，因为强类型、性能和治理能力更好。真实系统通常二者并存。

---

## 【文本块 38】Q：RPC 为什么需要序列化和协议？

远程调用要在网络上传输数据，内存里的对象不能直接跨进程传。

所以需要序列化：

> 把对象转换成字节流。

也需要协议：

> 规定这些字节怎么组织，怎么区分请求和响应，怎么标识服务、方法、状态码、metadata、body 长度等。

序列化关注：

* 性能
* 体积
* 兼容性
* 跨语言
* 可读性
* 安全性

协议关注：

* 请求响应格式
* 消息边界
* 多路复用
* 流控
* 错误码
* 元数据
* 心跳
* 超时

面试里可以说：

> RPC 性能不只是网络快不快，还和协议设计、序列化效率、连接复用、压缩、负载均衡和超时重试策略有关。

---

# 十四、RPC 治理

## 【文本块 39】Q：RPC 调用为什么必须设置超时？

因为远程调用不可靠。

下游可能慢、挂、网络抖动、连接池满。
如果没有超时，上游 goroutine 会一直等待，连接池和资源慢慢耗尽，最后引发雪崩。

Go 里一般通过 context 控制超时：

```go
ctx, cancel := context.WithTimeout(ctx, 200*time.Millisecond)
defer cancel()
```

然后把 ctx 传给 RPC client。

面试里可以说：

> 所有 RPC 调用都必须有 deadline。超时不是为了让请求失败，而是为了给系统设置资源边界，避免一个慢下游拖死整个上游。

---

## 【文本块 40】Q：RPC 重试有什么风险？

重试可以提高偶发失败下的成功率，但也会放大故障。

风险包括：

* 非幂等接口重复执行
* 重复扣款
* 重复下单
* 重复发券
* 下游已经很慢，重试进一步放大流量
* 上游超时后重试，导致请求风暴
* 多层服务都重试，形成指数级放大

所以重试必须满足：

* 接口幂等
* 有最大次数
* 有总超时预算
* 有退避策略
* 只对特定错误重试
* 不跨越业务语义重试
* 配合熔断和限流

面试里可以说：

> RPC 重试不是越多越好。读接口或天然幂等接口可以有限重试；写接口必须带幂等号，否则重试可能造成重复业务动作。

---

## 【代码块 7】Go RPC 调用 wrapper：timeout + retry 思路

```go
package main

import (
	"context"
	"errors"
	"fmt"
	"time"
)

type RPCFunc[T any] func(ctx context.Context) (T, error)

func CallWithRetry[T any](
	ctx context.Context,
	timeout time.Duration,
	maxRetry int,
	fn RPCFunc[T],
) (T, error) {
	var zero T
	var lastErr error

	for i := 0; i <= maxRetry; i++ {
		callCtx, cancel := context.WithTimeout(ctx, timeout)
		result, err := fn(callCtx)
		cancel()

		if err == nil {
			return result, nil
		}

		lastErr = err

		if !isRetryable(err) {
			return zero, err
		}

		select {
		case <-ctx.Done():
			return zero, ctx.Err()
		case <-time.After(time.Duration(i+1) * 50 * time.Millisecond):
		}
	}

	return zero, fmt.Errorf("rpc failed after retry: %w", lastErr)
}

func isRetryable(err error) bool {
	if errors.Is(err, context.Canceled) {
		return false
	}
	return true
}
```

---

## 【文本块 41】代码解释

这段代码是一个简化的 RPC 调用 wrapper。

它体现几个关键点：

每次调用都有 timeout。
重试次数有限。
重试之间有简单退避。
遇到不可重试错误直接返回。
外层 ctx 取消后立即停止。

真实项目里还要补充：

* 按错误码判断是否可重试
* 总耗时预算
* 熔断器
* 限流
* 指标统计
* trace span
* 日志采样
* 幂等号
* 负载均衡

---

# 十五、gRPC 基础

## 【文本块 42】Q：什么是 gRPC？

gRPC 是一个高性能 RPC 框架，Go 生态里非常常见。

它通常具有几个特点：

* 使用 Protobuf 定义接口和消息
* 基于 HTTP/2
* 支持多路复用
* 支持流式调用
* 强类型代码生成
* 跨语言
* 支持 metadata
* 支持 deadline
* 支持 interceptor
* 和服务发现、负载均衡、trace、metrics 容易集成

面试里可以这样回答：

> gRPC 是基于 HTTP/2 和 Protobuf 的 RPC 框架，适合内部服务间强类型、高性能、跨语言调用。它不仅是通信协议，还提供 deadline、metadata、stream、interceptor 等工程能力。

---

## 【文本块 43】Q：Protobuf 有什么优势？

Protobuf 的优势：

第一，体积小。
它是二进制编码，通常比 JSON 更紧凑。

第二，性能好。
序列化和反序列化速度通常比文本 JSON 更好。

第三，强类型。
接口字段在 proto 文件里定义，生成客户端和服务端代码，编译期能发现很多错误。

第四，跨语言。
同一个 proto 可以生成 Go、Java、Python、C++ 等语言代码。

第五，兼容演进。
通过字段编号和 optional 机制，可以支持接口字段向前兼容。

缺点：

* 可读性不如 JSON
* 调试不如 REST 方便
* 需要生成代码
* 字段编号不能随便改
* 对外开放 API 时门槛更高

面试里可以说：

> Protobuf 更适合内部服务间高频调用，JSON 更适合对外接口和调试友好的场景。

---

## 【代码块 8】proto 文件示例

```proto
syntax = "proto3";

package user.v1;

option go_package = "example.com/project/api/user/v1;userv1";

service UserService {
  rpc GetUser(GetUserRequest) returns (GetUserResponse);
  rpc ListUsers(ListUsersRequest) returns (ListUsersResponse);
}

message GetUserRequest {
  int64 user_id = 1;
}

message GetUserResponse {
  int64 user_id = 1;
  string name = 2;
  string email = 3;
}

message ListUsersRequest {
  int32 page = 1;
  int32 page_size = 2;
}

message ListUsersResponse {
  repeated GetUserResponse users = 1;
}
```

---

## 【文本块 44】代码解释

proto 文件定义了：

* package
* Go 代码生成路径
* service
* rpc 方法
* request message
* response message

字段后面的数字是字段编号，不能随便改。
删除字段时，也要避免复用老字段编号，否则可能导致兼容性问题。

面试里要记住：

> Protobuf 的字段编号是二进制编码和兼容性的关键，不要随便修改或复用。

---

# 十六、gRPC 调用类型

## 【文本块 45】Q：gRPC 支持哪些调用类型？

gRPC 支持四种调用方式。

第一，Unary RPC。
一请求一响应，最常见。

```text
client -> request
server -> response
```

第二，Server Streaming。
客户端发一个请求，服务端返回流式响应。适合下载、推送、日志流。

第三，Client Streaming。
客户端持续发送多条消息，服务端最后返回一个响应。适合上传、批量写入。

第四，Bidirectional Streaming。
客户端和服务端双向流式通信。适合 IM、实时协作、长连接、实时状态同步。

面试里可以说：

> 大多数业务接口用 Unary；大量数据下发用 Server Streaming；客户端批量上传用 Client Streaming；实时双向通信用 Bidirectional Streaming。

---

## 【文本块 46】Q：gRPC 的 deadline 是什么？

deadline 表示一次 RPC 调用的截止时间。

在 Go 里，通常通过 context timeout 传递：

```go
ctx, cancel := context.WithTimeout(context.Background(), 200*time.Millisecond)
defer cancel()
```

gRPC 会把 deadline 透传给服务端。
服务端可以通过 ctx 感知调用是否超时或取消。

如果超时，客户端通常会得到 DeadlineExceeded 错误。

面试里可以说：

> gRPC deadline 是服务调用的资源边界。客户端必须设置 deadline，服务端也应该尊重 ctx.Done，不要在请求已经取消后继续浪费资源。

---

## 【代码块 9】gRPC 客户端 deadline 示意代码

```go
package main

import (
	"context"
	"fmt"
	"time"
)

type GetUserRequest struct {
	UserID int64
}

type GetUserResponse struct {
	UserID int64
	Name   string
}

type UserClient interface {
	GetUser(ctx context.Context, req *GetUserRequest) (*GetUserResponse, error)
}

func QueryUser(ctx context.Context, client UserClient, userID int64) (*GetUserResponse, error) {
	callCtx, cancel := context.WithTimeout(ctx, 200*time.Millisecond)
	defer cancel()

	resp, err := client.GetUser(callCtx, &GetUserRequest{
		UserID: userID,
	})
	if err != nil {
		return nil, fmt.Errorf("grpc get user: %w", err)
	}

	return resp, nil
}
```

---

## 【文本块 47】代码解释

这里没有依赖真实生成的 pb 代码，而是用接口表达 gRPC client 的调用方式。

重点是：

* 调用前设置 timeout
* ctx 向下传递
* 返回错误时包装上下文
* 不在函数内部用 context.Background 切断上游 deadline

Go 微服务里最忌讳：

```go
context.Background()
```

在业务链路中随便使用。这样会让超时、取消和 trace 全部断掉。

---

# 十七、gRPC Metadata 与 Interceptor

## 【文本块 48】Q：gRPC metadata 是什么？

metadata 可以理解成 gRPC 里的请求头。

它常用于传递：

* trace_id
* request_id
* user_id
* token
* 灰度标识
* 租户 ID
* 语言区域
* 调用来源

HTTP 有 Header。
gRPC 有 Metadata。

Go 里可以从 incoming context 中读取 metadata，也可以向 outgoing context 写入 metadata。

面试里可以说：

> metadata 适合放请求级元信息，不适合放大对象或业务主体数据。跨服务传 trace_id、鉴权信息、灰度标识时非常常见。

---

## 【文本块 49】Q：gRPC interceptor 有什么用？

interceptor 类似 HTTP middleware。

它可以在 RPC 调用前后统一做横切逻辑，比如：

* 日志
* panic recover
* 鉴权
* 参数校验
* metrics
* tracing
* timeout
* retry
* rate limit
* circuit breaker
* 错误码转换

服务端有 server interceptor。
客户端有 client interceptor。
Unary 和 Stream 也有不同 interceptor。

面试里可以说：

> gRPC interceptor 是服务治理的核心扩展点。不要把日志、鉴权、trace、metrics 散落在每个业务方法里，应该通过 interceptor 统一处理。

---

## 【代码块 10】gRPC Unary Server Interceptor 示意代码

```go
package main

import (
	"context"
	"log"
	"time"
)

type UnaryHandler func(ctx context.Context, req any) (any, error)

type UnaryServerInfo struct {
	FullMethod string
}

type UnaryServerInterceptor func(
	ctx context.Context,
	req any,
	info *UnaryServerInfo,
	handler UnaryHandler,
) (any, error)

func LoggingInterceptor(
	ctx context.Context,
	req any,
	info *UnaryServerInfo,
	handler UnaryHandler,
) (any, error) {
	start := time.Now()

	resp, err := handler(ctx, req)

	cost := time.Since(start)
	if err != nil {
		log.Printf("method=%s cost=%s err=%v", info.FullMethod, cost, err)
	} else {
		log.Printf("method=%s cost=%s", info.FullMethod, cost)
	}

	return resp, err
}
```

---

## 【文本块 50】代码解释

这段代码是 gRPC interceptor 的简化形态，方便理解。

真实 gRPC 里会使用 `grpc.UnaryServerInterceptor` 类型。
但核心思想一样：

请求进入业务 handler 前，先经过 interceptor。
业务 handler 执行完后，再回到 interceptor。
这样可以统一记录耗时、错误、trace、metrics。

---

# 十八、gRPC 错误码

## 【文本块 51】Q：gRPC 错误怎么设计？

gRPC 有标准 status code，比如：

OK
Canceled
Unknown
InvalidArgument
DeadlineExceeded
NotFound
AlreadyExists
PermissionDenied
ResourceExhausted
FailedPrecondition
Aborted
OutOfRange
Unimplemented
Internal
Unavailable
DataLoss
Unauthenticated

工程里应该把业务错误和系统错误映射到合适的 gRPC code。

例如：

参数错误 -> InvalidArgument
未登录 -> Unauthenticated
无权限 -> PermissionDenied
资源不存在 -> NotFound
重复创建 -> AlreadyExists
限流 -> ResourceExhausted
下游不可用 -> Unavailable
内部异常 -> Internal
超时 -> DeadlineExceeded

面试里可以说：

> gRPC 不应该所有错误都返回 Internal。错误码要稳定、语义清晰，方便调用方做分支处理，也方便监控统计。

---

# 十九、Go 网络框架总览

## 【文本块 52】Q：Go 常见 Web/网络框架有哪些？

可以按层次区分。

第一，标准库 net/http。
Go 官方 HTTP 基础库，稳定、简单、性能足够好。很多框架底层也依赖它。

第二，Web 框架。
Gin、Echo、Fiber、Hertz 等，主要提供路由、中间件、参数绑定、校验、错误处理、分组路由等能力。

第三，微服务框架。
Kratos、go-zero、Kitex 等，通常更关注服务治理、RPC、配置、注册发现、日志、metrics、tracing、错误码、代码生成。

第四，RPC 框架。
gRPC、Kitex 等，主要解决服务间通信。

第五，事件驱动网络框架。
例如 gnet 这类更接近 Go 里的“Netty 风格”网络框架，适合某些极端连接数或自定义协议场景，但普通业务服务不一定需要。

面试里可以说：

> 普通 HTTP 服务用 net/http、Gin、Echo、Hertz 都可以；内部 RPC 用 gRPC、Kitex；完整微服务脚手架可以考虑 Kratos 或 go-zero。选型看团队生态、治理需求、性能需求和维护成本，而不是盲目追求框架名气。

---

## 【文本块 53】Q：Go 里为什么不常说“Go 版 Netty”？

Java 里 Netty 很重要，是因为原生 NIO 编程复杂。Netty 把 Selector、Channel、EventLoop、Pipeline、ByteBuf、编解码、心跳、粘包拆包等工程细节封装好了。

Go 的情况不同。

Go 标准库提供了非常好用的 net 和 net/http。
Go runtime 底层有 netpoller。
开发者可以用同步阻塞风格写代码：

```go
conn.Read(buf)
```

但底层不会真的让一个 OS 线程被每个连接长期阻塞。goroutine 等待网络 IO 时会被挂起，线程可以去执行其他 goroutine。

所以 Go 里普通网络服务不需要像 Java 一样强依赖 Netty。

面试里可以这样答：

> Java Netty 是对 NIO/Reactor 的工程化封装；Go 则把高并发网络 IO 很大程度封装进 runtime 和标准库里。Go 开发者通常用 goroutine-per-connection 的同步写法，就能获得比较好的并发能力。只有自定义协议、极端长连接、事件驱动优化场景，才会考虑更底层的网络框架。

---

# 二十、net/http 与 middleware

## 【文本块 54】Q：Go net/http 的核心模型是什么？

net/http 的核心非常简单：

```go
type Handler interface {
    ServeHTTP(ResponseWriter, *Request)
}
```

只要实现了 ServeHTTP，就可以处理 HTTP 请求。

常见的 handler function：

```go
func(w http.ResponseWriter, r *http.Request)
```

Go Web 框架本质上也都围绕这个模型扩展：

* 路由匹配
* 中间件链
* 参数解析
* JSON 响应
* 错误处理
* recover
* 日志
* metrics
* trace

面试里可以说：

> net/http 的核心是 Handler 接口和 ServeHTTP 方法。Go Web 框架的大部分能力，本质上是在标准库 handler 模型上增加路由和 middleware 体系。

---

## 【代码块 11】Go HTTP middleware 链

```go
package main

import (
	"log"
	"net/http"
	"time"
)

type Middleware func(http.Handler) http.Handler

func Chain(h http.Handler, middlewares ...Middleware) http.Handler {
	for i := len(middlewares) - 1; i >= 0; i-- {
		h = middlewares[i](h)
	}
	return h
}

func Logging() Middleware {
	return func(next http.Handler) http.Handler {
		return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
			start := time.Now()
			next.ServeHTTP(w, r)
			log.Printf("method=%s path=%s cost=%s", r.Method, r.URL.Path, time.Since(start))
		})
	}
}

func Recover() Middleware {
	return func(next http.Handler) http.Handler {
		return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
			defer func() {
				if rec := recover(); rec != nil {
					log.Printf("panic: %v", rec)
					http.Error(w, "internal server error", http.StatusInternalServerError)
				}
			}()
			next.ServeHTTP(w, r)
		})
	}
}

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
		_, _ = w.Write([]byte("hello"))
	})

	handler := Chain(mux, Recover(), Logging())

	_ = http.ListenAndServe(":8080", handler)
}
```

---

## 【文本块 55】代码解释

这个 middleware 链体现了 Go Web 框架的核心思想：

请求进入时先经过 Recover。
再经过 Logging。
最后进入真正业务 handler。

这类 middleware 还可以扩展：

* trace_id
* auth
* CORS
* rate limit
* request body limit
* metrics
* timeout
* pprof
* error mapping

Go 框架的很多能力，本质上就是把这些横切逻辑封装得更好。

---

# 二十一、Gin、Echo、Fiber、Hertz 怎么看

## 【文本块 56】Q：Go Web 框架怎么选？

可以从几个维度看：

第一，是否需要标准库兼容。
Gin、Echo、Hertz 和 net/http 生态结合方式不同。标准库兼容性越好，接入 pprof、otel、middleware、http client/server 工具会更自然。

第二，团队熟悉度。
框架再好，团队不熟也会增加维护成本。

第三，性能需求。
绝大多数业务瓶颈不在路由框架，而在 DB、Redis、RPC、锁、序列化、下游依赖。不要过度迷信框架性能榜。

第四，治理能力。
如果只是简单 HTTP API，Gin/Echo 足够。
如果要完整微服务治理、代码生成、RPC、配置、注册发现，可以看 Kratos、go-zero、Hertz/Kitex 等体系。

第五，生态和维护。
文档、社区、插件、升级稳定性都很重要。

面试里可以这样回答：

> Go Web 框架选型我会优先看团队熟悉度、标准库兼容性、生态和治理需求。普通业务服务 Gin/Echo/net/http 都够用；性能瓶颈通常不在框架路由，而在数据库、缓存、RPC 和业务逻辑。微服务体系更关注注册发现、配置、日志、trace、metrics 和错误码规范。

---

# 二十二、Go 自定义 TCP 协议与粘包拆包

## 【文本块 57】Q：TCP 为什么会有粘包拆包问题？

TCP 是字节流协议，不保留应用层消息边界。

你发送两次：

```text
hello
world
```

接收端可能一次读到：

```text
helloworld
```

也可能分两次读到：

```text
hel
loworld
```

这不是 TCP 出错，而是 TCP 的正常语义。

应用层必须自己定义协议边界。

常见解决方式：

第一，固定长度。
每个消息固定 N 字节。

第二，分隔符。
比如以 `\n` 作为一条消息结束。

第三，长度字段。
消息头里写 body 长度，接收端先读 header，再按长度读 body。

第四，TLV。
Type-Length-Value，适合更复杂协议。

面试里可以说：

> TCP 是流，不是消息。粘包拆包的本质是应用层没有消息边界。解决方式就是在应用协议里定义边界，比如固定长度、分隔符、长度字段或 TLV。

---

## 【代码块 12】Go TCP 长度字段协议编码示例

```go
package main

import (
	"bytes"
	"encoding/binary"
	"fmt"
)

func EncodeMessage(body []byte) ([]byte, error) {
	var buf bytes.Buffer

	length := uint32(len(body))
	if err := binary.Write(&buf, binary.BigEndian, length); err != nil {
		return nil, err
	}

	if _, err := buf.Write(body); err != nil {
		return nil, err
	}

	return buf.Bytes(), nil
}

func DecodeMessage(packet []byte) ([]byte, error) {
	if len(packet) < 4 {
		return nil, fmt.Errorf("packet too short")
	}

	length := binary.BigEndian.Uint32(packet[:4])
	if len(packet[4:]) < int(length) {
		return nil, fmt.Errorf("body not complete")
	}

	return packet[4 : 4+length], nil
}

func main() {
	packet, _ := EncodeMessage([]byte("hello tcp"))
	body, _ := DecodeMessage(packet)

	fmt.Println(string(body))
}
```

---

## 【文本块 58】代码解释

这个协议格式是：

```text
4 字节长度 + body
```

接收端先读 4 字节，知道 body 长度。
再继续读取指定长度的 body。

真实 TCP 服务里，Read 可能读到半包，所以还需要循环读取、缓冲区累积和状态机解析。

这也是 Netty 这类框架在 Java 里很重要的原因：它帮你封装了编解码、粘包拆包、Pipeline 和事件处理。Go 里如果写自定义 TCP 协议，也同样要处理这些问题，只是可以用 goroutine 和标准库更直接地组织代码。

---

# 二十三、项目里怎么讲这一章

## 【文本块 59】Q：如果项目里用了 MQ，面试怎么讲更像真实做过？

可以按四步讲。

第一，场景。
比如下单后异步发通知、发积分、同步 ES、写埋点，或者秒杀场景用 MQ 削峰。

第二，为什么用。
同步调用多个下游会拖慢主链路，也会让订单服务强依赖下游。MQ 可以解耦、削峰、异步化。

第三，可靠性怎么做。
生产端发送确认或本地消息表；Broker 端持久化和副本；消费端手动 ack，业务成功后确认；失败重试，超过次数进死信；消费端幂等。

第四，踩过什么坑。
比如重复消费、消息积压、消费者并发过高打爆 DB、消费失败重试风暴、顺序消息吞吐低、ES 同步延迟。

---

## 【文本块 60】Q：如果项目里用了 ES，面试怎么讲？

可以这样组织：

第一，场景。
商品搜索、日志检索、订单后台复杂查询、文章搜索、运营报表。

第二，为什么不用 MySQL。
MySQL 适合事务和精确查询，不适合大规模全文检索、分词、相关性评分和复杂聚合。

第三，索引怎么设计。
哪些字段 text，哪些 keyword，哪些用于过滤、排序、聚合，分片和副本如何设置。

第四，数据怎么同步。
MySQL 是主库，ES 是查询索引，通过 MQ、Outbox 或 binlog CDC 同步，接受最终一致。

第五，踩过什么坑。
比如 mapping 设计错、深分页慢、分词不准、索引膨胀、同步延迟、重复写入、ES 查询拖慢集群。

---

## 【文本块 61】Q：如果项目里用了 gRPC，面试怎么讲？

可以这样组织：

第一，场景。
内部服务之间调用，比如订单服务调用用户服务、库存服务、支付服务。

第二，为什么选 gRPC。
强类型、Protobuf 体积小、HTTP/2 多路复用、跨语言、支持 deadline、metadata、interceptor、stream。

第三，治理怎么做。
超时、重试、负载均衡、服务发现、限流、熔断、trace、metrics、错误码规范。

第四，踩过什么坑。
比如没设置 deadline 导致请求堆积；非幂等接口重试导致重复操作；metadata 没透传导致 trace 断链；proto 字段编号乱改导致兼容问题。

---

# 二十四、本章速记版

## 【文本块 62】MQ 速记

MQ 价值：

* 解耦
* 削峰
* 异步化
* 最终一致
* 失败重试
* 事件驱动

MQ 风险：

* 消息丢失
* 重复消费
* 顺序问题
* 消息积压
* 死信处理
* 数据最终一致
* 排查复杂

可靠性三段：

* 生产端确认
* Broker 持久化和副本
* 消费端业务成功后 ack

消费端必须幂等。
重试必须有边界。
死信必须有人处理。
积压必须有监控。
异步化必须有补偿。

---

## 【文本块 63】Kafka/RocketMQ/RabbitMQ 速记

Kafka：

* 高吞吐
* 日志流
* 埋点流
* 大数据链路
* 顺序写
* Page Cache
* 零拷贝
* 分区并行

RocketMQ：

* 业务消息
* 顺序消息
* 延迟消息
* 事务消息
* 重试死信
* 订单支付库存场景

RabbitMQ：

* AMQP
* Exchange 路由灵活
* Direct/Fanout/Topic/Headers
* Confirm
* 手动 ack
* TTL
* 死信队列
* 中后台异步任务

选型一句话：

> 大吞吐日志流看 Kafka，核心业务消息看 RocketMQ，灵活路由和传统业务队列看 RabbitMQ。

---

## 【文本块 64】ES 速记

ES 是分布式全文检索和分析引擎。

适合：

* 商品搜索
* 日志检索
* 文章搜索
* 复杂查询
* 聚合分析

核心：

* 倒排索引
* 分词器
* mapping
* shard
* replica
* refresh
* search_after
* bulk
* alias

MySQL 是主存储。
ES 是查询索引。
两者通过 MQ、Outbox、binlog CDC 同步。
ES 是近实时，不适合强一致主链路查询。
深分页要避免，优先 search_after。
mapping 设计要贴合查询场景。

---

## 【文本块 65】RPC/gRPC 速记

RPC 是远程过程调用。

核心能力：

* 协议
* 序列化
* 网络通信
* 服务发现
* 负载均衡
* 超时
* 重试
* 熔断
* 限流
* 监控
* 追踪

gRPC：

* Protobuf
* HTTP/2
* 强类型
* 跨语言
* Unary
* Server Streaming
* Client Streaming
* Bidirectional Streaming
* Metadata
* Deadline
* Interceptor
* Status Code

所有 RPC 必须设置超时。
重试必须幂等。
metadata 透传 trace。
interceptor 做治理。
错误码要语义稳定。

---

## 【文本块 66】Go 网络框架速记

net/http 是 Go HTTP 基础。
Gin/Echo/Fiber/Hertz 提供 Web 开发能力。
Kratos/go-zero 更偏微服务脚手架。
gRPC/Kitex 更偏内部 RPC。
gnet 更偏事件驱动网络框架。

Java Netty 重要，是因为原生 NIO 工程复杂。
Go 不常依赖“Go 版 Netty”，因为 runtime netpoller + goroutine + net/http 已经封装了大量网络 IO 复杂度。

Go 网络服务重点：

* context 超时取消
* middleware
* recover
* pprof
* metrics
* trace
* graceful shutdown
* 连接池
* 限流
* backpressure

---

# 二十五、本章最终面试回答模板

## 【文本块 67】综合回答模板

如果面试官让我整体讲消息队列、ES、RPC、gRPC 和 Go 网络框架，我会这样回答：

消息队列主要解决解耦、削峰和异步化。比如订单创建后，通知、积分、埋点、搜索索引同步这些动作没必要阻塞主链路，可以通过 MQ 异步处理。但引入 MQ 后必须解决可靠投递、重复消费、顺序消息、消息积压、死信处理和最终一致。可靠性要分生产端、Broker、消费端三段看：生产端要确认发送成功，Broker 要持久化和副本，消费端要业务成功后再 ack。由于 MQ 通常是至少一次投递，所以消费端必须幂等。

MQ 选型上，我会根据业务场景判断。Kafka 更适合高吞吐日志流、埋点流和大数据链路；RocketMQ 更适合订单、支付、库存这类核心业务消息，因为顺序消息、延迟消息、事务消息、重试死信这些能力比较贴近业务；RabbitMQ 更适合中后台系统、通知分发、审批流、异步任务这些需要灵活路由但吞吐不是极致的场景。真正选型不是谁功能最多，而是谁更适合当前业务、团队和运维能力。

Elasticsearch 是分布式全文检索和分析引擎，适合商品搜索、日志检索、文章搜索、复杂筛选和聚合分析。它的核心是倒排索引，通过从 term 到文档列表的映射加速全文搜索。MySQL 更适合事务和精确查询，ES 更适合搜索和分析，所以我一般不会把 ES 当主库，而是把 MySQL 作为事实源，ES 作为查询侧索引。数据同步可以通过 MQ、Outbox 或 binlog CDC 做最终一致。ES 是近实时系统，写入后要经过 refresh 才能被搜索到，所以业务要能接受短暂延迟。

RPC 的核心是让服务间远程调用像本地方法一样使用，但它底层包含协议、序列化、网络通信、服务发现、负载均衡、超时、重试、熔断、限流、监控和链路追踪等能力。gRPC 是 Go 生态里常见的 RPC 框架，它基于 HTTP/2 和 Protobuf，支持强类型代码生成、metadata、deadline、interceptor 和流式调用。使用 gRPC 时，所有调用都要设置 deadline，metadata 要透传 trace 信息，interceptor 用来统一做日志、鉴权、metrics、tracing 和错误码转换。重试必须建立在幂等基础上，否则可能导致重复下单、重复扣款等问题。

Go 网络框架方面，普通 HTTP 服务可以直接用 net/http，也可以用 Gin、Echo、Fiber、Hertz 这类框架提升开发效率。微服务体系可以考虑 Kratos、go-zero、gRPC、Kitex 等。和 Java 里常讲 Netty 不同，Go runtime 底层已经通过 netpoller 和 goroutine 把高并发网络 IO 封装得比较好，开发者可以用同步阻塞风格写代码，但等待网络 IO 的 goroutine 不会长期占住 OS 线程。所以普通业务服务不一定需要“Go 版 Netty”，只有自定义协议、极端长连接或事件驱动优化场景才会考虑更底层网络框架。

一句话总结：MQ 负责异步事件和削峰，ES 负责搜索和分析，RPC/gRPC 负责服务间强类型高效通信，Go Web/网络框架负责请求接入和治理扩展。真正工程化的回答不能停留在“用了什么中间件”，而要讲清楚为什么用、怎么保证可靠、怎么处理超时重试、怎么做幂等、怎么监控排查，以及出了问题如何补偿恢复。
